# ViTrueVoice — Pipeline Tiền Xử Lý Dữ Liệu & Sinh Spoof Trên Kaggle GPU

Hệ thống phát hiện giọng nói giả tiếng Việt qua kênh thoại nén.
Notebook thực thi trọn vẹn:
1. Chuẩn hóa âm thanh Bonafide 16kHz mono PCM.
2. Tự động sinh dữ liệu Spoof qua F5-TTS và VieNeu-TTS (Adaptive mode) kèm nén viễn thông G.711 / AMR-NB.
3. Lập bảng manifest 17 cột theo `DataColunm_Plan.docx` và phân vùng Speaker-Disjoint (Train, Dev, Test).
4. Đóng gói kết quả về file ZIP để tải về máy local qua `manage_kaggle.py output`.


In [ ]:
# 1. Cài đặt các thư viện cần thiết trên Kaggle GPU
import subprocess
import sys
import os

print('📦 Đang cài đặt thư viện phụ thuộc (F5-TTS, VieNeu-TTS, SpeechBrain, Codecs)...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'f5-tts', 'vieneu', 'speechbrain', 'jiwer', 'soundfile', 'librosa', 'pyloudnorm'], check=False)
subprocess.run(['apt-get', 'update', '-qq'], check=False)
subprocess.run(['apt-get', 'install', '-y', '-qq', 'ffmpeg', 'libopencore-amrnb-dev'], check=False)

# Kiểm tra bộ giải mã ffmpeg
res = subprocess.run(['ffmpeg', '-encoders'], capture_output=True, text=True)
has_amr = 'libopencore_amrnb' in res.stdout
has_alaw = 'pcm_alaw' in res.stdout
print(f'🔍 Kiểm tra FFmpeg: AMR-NB (libopencore_amrnb)={has_amr}, G.711 A-law (pcm_alaw)={has_alaw}')
if not has_amr:
    print('⚠️ CẢNH BÁO: ffmpeg chưa có libopencore_amrnb. Thử cài đặt gói codecs bổ sung...')
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'libavcodec-extra'], check=False)

import torch
print('=' * 60)
print(f'PyTorch Version : {torch.__version__}')
print(f'CUDA Available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Device      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM            : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')
else:
    print('⚠️ CẢNH BÁO: Không có GPU! Pipeline sẽ chạy rất chậm.')
print('=' * 60)


In [ ]:
# 2. Khởi tạo package vitruevoice trên Kaggle container
import base64
import zlib
import json
import sys
from pathlib import Path

PKG_B64 = '...'

pkg_data = json.loads(zlib.decompress(base64.b64decode(PKG_B64.encode('ascii'))).decode('utf-8'))
working_dir = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('./out')
pkg_root = working_dir / 'vitruevoice'
pkg_root.mkdir(parents=True, exist_ok=True)

for rel_path, content in pkg_data.items():
    file_path = pkg_root / rel_path
    file_path.parent.mkdir(parents=True, exist_ok=True)
    file_path.write_text(content, encoding='utf-8')

if str(working_dir) not in sys.path:
    sys.path.insert(0, str(working_dir))

print(f'✅ Đã khởi tạo thành công package vitruevoice tại: {pkg_root} ({len(pkg_data)} files)')


In [ ]:
# 3. Tiền xử lý Bonafide Audio: 16kHz Mono 16-bit PCM & trích xuất Ref 3.5s-7.5s
import json
import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import subprocess
from pathlib import Path
from vitruevoice.batch_processor import DatasetPartitioner

# ==============================================================
# CẤU HÌNH THỰC THI (SMOKE TEST VS FULL RUN)
# ==============================================================
SMOKE_TEST = False  # False: Chạy toàn quy mô trên toàn bộ VIVOS + Common Voice
MAX_VIVOS_SPEAKERS = 65  # Toàn bộ 65 người nói VIVOS (46 train, 19 test)
MAX_CV_SPEAKERS = 25     # Tối đa 25 người nói chất lượng cao nhất từ Common Voice
MAX_PROMPTS_PER_SPK = 5  # 5 câu spoof / người nói (sinh 30 mẫu spoof / spk qua 2 TTS x 3 codec)

print(f'⚙️ Chế độ thực thi: {"SMOKE TEST (thử nghiệm nhanh)" if SMOKE_TEST else "FULL RUN (toàn quy mô VIVOS + Common Voice)"}')

data_root = working_dir / 'data'
raw_out_dir = data_root / 'raw'
ref_out_dir = data_root / 'bonafide_ref'
manifests_dir = data_root / 'manifests'
checkpoints_dir = working_dir / 'checkpoints'

raw_out_dir.mkdir(parents=True, exist_ok=True)
ref_out_dir.mkdir(parents=True, exist_ok=True)
manifests_dir.mkdir(parents=True, exist_ok=True)
checkpoints_dir.mkdir(parents=True, exist_ok=True)

# --- 1. TÌM KIẾM DATASET VIVOS & COMMON VOICE TRÊN KAGGLE ---
input_roots = [Path('/kaggle/input'), Path('./data')]

# A. Tìm VIVOS
vivos_root = None
for r in input_roots:
    if r.exists():
        cand = list(r.glob('**/vivos')) + list(r.glob('**/prompts.txt'))
        if cand:
            vivos_root = cand[0].parent if cand[0].is_dir() else cand[0].parent.parent
            break

if vivos_root:
    print(f'📂 Tìm thấy dataset VIVOS tại: {vivos_root}')
else:
    print('⚠️ Chưa tìm thấy VIVOS tại /kaggle/input! Kiểm tra kernel-metadata.json có slug vivos.')

# B. Tìm Common Voice
cv_tsv = None
for r in input_roots:
    if r.exists():
        cand = list(r.glob('**/validated.tsv'))
        if cand:
            cv_tsv = cand[0]
            break

if cv_tsv:
    print(f'📂 Tìm thấy dataset Common Voice tại: {cv_tsv}')
else:
    print('ℹ️ Không tìm thấy validated.tsv của Common Voice trong /kaggle/input.')

# Hàm tiền xử lý sóng âm: resample 16kHz mono, cắt lặng top_db=25 với 150ms đệm, chuẩn hóa loudness
def standardize_audio(src_path, dst_path, target_sr=16000):
    wav, sr = librosa.load(src_path, sr=target_sr, mono=True)
    if len(wav) > 512:
        _, idx = librosa.effects.trim(wav, top_db=25, frame_length=512, hop_length=160)
        margin = int(target_sr * 0.15)  # 150ms đệm an toàn
        start = max(0, idx[0] - margin)
        end = min(len(wav), idx[1] + margin)
        if end > start:
            wav = wav[start:end]
    # Peak Headroom scaling <= 0.95 (tránh clipping)
    peak = np.max(np.abs(wav))
    if peak > 1e-6:
        wav = wav * (0.95 / max(peak, 1.0))
    Path(dst_path).parent.mkdir(parents=True, exist_ok=True)
    sf.write(str(dst_path), wav, target_sr, subtype='PCM_16')
    dur = len(wav) / target_sr
    return round(dur, 4)

# Nạp VIVOS Prompts (chuẩn hóa chữ thường lowercase)
all_transcripts = {}
vivos_prompts = {}
if vivos_root:
    for p_file in vivos_root.rglob('prompts.txt'):
        with open(p_file, 'r', encoding='utf-8') as f:
            for line in f:
                parts = line.strip().split(maxsplit=1)
                if len(parts) == 2:
                    vivos_prompts[parts[0]] = parts[1].strip().lower()
                    all_transcripts[parts[0]] = parts[1].strip().lower()
print(f'📖 Đã nạp {len(vivos_prompts)} câu transcript từ VIVOS.')

# Nạp Common Voice Prompts & Clips
cv_prompts = {}
cv_speaker_files = {}
if cv_tsv:
    cand_clips = list(cv_tsv.parent.glob('**/clips'))
    if not cand_clips:
        cand_clips = list(cv_tsv.parent.parent.glob('**/clips'))
    clips_dir = cand_clips[0] if cand_clips else cv_tsv.parent
    print(f'📁 Thư mục clips Common Voice: {clips_dir}')

    cv_df = pd.read_csv(cv_tsv, sep='\t', low_memory=False)
    if 'up_votes' in cv_df.columns and 'down_votes' in cv_df.columns:
        valid_mask = (cv_df['up_votes'] >= cv_df['down_votes']) & (cv_df['sentence'].astype(str).str.len() > 10)
    else:
        valid_mask = cv_df['sentence'].astype(str).str.len() > 10
    valid_cv_df = cv_df[valid_mask]

    for _, row in valid_cv_df.iterrows():
        p_name = str(row['path'])
        clip_path = clips_dir / p_name
        if clip_path.exists():
            stem = clip_path.stem
            spk_id = f"CV_{str(row['client_id'])[:10]}"
            sent = str(row['sentence']).strip().lower()
            cv_prompts[stem] = sent
            all_transcripts[stem] = sent
            cv_speaker_files.setdefault(spk_id, []).append(clip_path)

    print(f'📖 Đã nạp {len(cv_prompts)} câu transcript và {len(cv_speaker_files)} người nói từ Common Voice.')

# Thu thập và phân loại người nói
bonafide_records = []
speaker_files_map = {}

# 1. Thêm người nói VIVOS
if vivos_root:
    wave_files = sorted(vivos_root.rglob('*.wav'))
    for wf in wave_files:
        spk = wf.parent.name
        speaker_files_map.setdefault(spk, []).append(wf)

# 2. Thêm người nói Common Voice (chỉ lấy người nói có >= 8 clips)
for spk, c_files in cv_speaker_files.items():
    if len(c_files) >= 8:
        speaker_files_map[spk] = c_files

# Lựa chọn danh sách người nói
vivos_spks = sorted([s for s in speaker_files_map if not s.startswith('CV_')])[:MAX_VIVOS_SPEAKERS]
cv_spks = sorted([s for s in speaker_files_map if s.startswith('CV_')], key=lambda s: len(speaker_files_map[s]), reverse=True)[:MAX_CV_SPEAKERS]

if SMOKE_TEST:
    selected_speakers = sorted(list(speaker_files_map.keys()))[:3]
    MAX_PROMPTS_PER_SPK = 3
else:
    selected_speakers = sorted(vivos_spks + cv_spks)

print(f'👥 Tổng số speaker được chọn: {len(selected_speakers)} (VIVOS: {len([s for s in selected_speakers if not s.startswith("CV_")])}, Common Voice: {len([s for s in selected_speakers if s.startswith("CV_")])})')
print(f'   - Danh sách mẫu: {selected_speakers[:10]} ...')

# Phân chia Speaker-Disjoint: 70% Train, 15% Dev, 15% Test
n_spk = len(selected_speakers)
n_train = max(1, min(int(round(n_spk * 0.70)), n_spk - 2))
n_dev = max(1, min(int(round(n_spk * 0.15)), n_spk - n_train - 1))

rng = np.random.RandomState(42)
shuffled_spks = np.array(selected_speakers)
rng.shuffle(shuffled_spks)

train_spks = set(shuffled_spks[:n_train])
dev_spks = set(shuffled_spks[n_train:n_train + n_dev])
test_spks = set(shuffled_spks[n_train + n_dev:])

print(f'📊 Phân chia Speaker-Disjoint: Train={len(train_spks)}, Dev={len(dev_spks)}, Test={len(test_spks)}')
assert train_spks.isdisjoint(dev_spks), 'Rò rỉ người nói giữa Train và Dev!'
assert train_spks.isdisjoint(test_spks), 'Rò rỉ người nói giữa Train và Test!'
assert dev_spks.isdisjoint(test_spks), 'Rò rỉ người nói giữa Dev và Test!'
assert len(train_spks) + len(dev_spks) + len(test_spks) == n_spk, 'Tổng số speaker không khớp!'

for spk in selected_speakers:
    raw_spk_dir = raw_out_dir / spk
    ref_spk_dir = ref_out_dir / spk
    raw_spk_dir.mkdir(parents=True, exist_ok=True)
    ref_spk_dir.mkdir(parents=True, exist_ok=True)

    spk_split = 'train' if spk in train_spks else ('dev' if spk in dev_spks else 'test')
    spk_source = 'common_voice' if spk.startswith('CV_') else 'vivos'
    files = speaker_files_map[spk]

    # Phân chia 3 tập rời rạc cho từng speaker: ref, enroll, test
    # Đọc nhanh duration của từng file để ưu tiên clip dải vàng 3.5s - 7.5s lên làm ref_cands
    file_durs = {}
    for f in files:
        try:
            info = sf.info(str(f))
            file_durs[str(f)] = info.duration
        except Exception:
            file_durs[str(f)] = 0.0

    ref_cands, enroll_cands, eval_cands = DatasetPartitioner.partition_speaker_files(
        files, n_ref_cands=3, n_enroll=4, file_durations=file_durs
    )

    for f in files:
        stem = f.stem
        role = DatasetPartitioner.assign_utterance_role(f, ref_cands, enroll_cands)
        dst_wav = raw_spk_dir / f'{stem}.wav'
        dur = standardize_audio(f, dst_wav)
        if role == 'ref':
            ref_wav = ref_spk_dir / f'{stem}.wav'
            sf.write(str(ref_wav), sf.read(str(dst_wav))[0], 16000)

        transcript = all_transcripts.get(stem, '').strip().lower()
        bonafide_records.append({
            'utt_id': stem,
            'audio_path': f'data/raw/{spk}/{stem}.wav',
            'speaker_id': spk,
            'source': spk_source,
            'label': 'bonafide',
            'role': role,
            'tts_generator': 'none',
            'codec': 'clean_16khz',
            'split': spk_split,
            'duration_sec': dur,
            'transcript': transcript,
            'raw_wer': None,
            'raw_cer': None,
            'ecapa_sim': None,
            'wavlm_sim': None,
            'derived_tier': 'BonaFide',
            'seed': 100,
        })

bonafide_df = pd.DataFrame(bonafide_records)
bonafide_csv = manifests_dir / 'bonafide_manifest.csv'
bonafide_df.to_csv(bonafide_csv, index=False, encoding='utf-8')
print(f'✅ Đã lưu {len(bonafide_df)} mẫu Bona fide chuẩn hóa tại: {bonafide_csv}')


In [ ]:
# 4. Sinh giọng Spoof qua Dual Pipeline F5-TTS / VieNeu-TTS (kèm Checkpoint & Resume)
from vitruevoice.batch_processor import BatchDatasetProcessor
from vitruevoice.pipeline import ViTrueVoiceDualPipeline, PipelineMode
from vitruevoice.evaluator import MultiGateEvaluator

device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
if 'processor' not in globals() or processor is None:
    print(f'🚀 Đang khởi tạo Dual Pipeline và Evaluator trên thiết bị: {device}...')
    evaluator = MultiGateEvaluator(device=device)
    pipeline = ViTrueVoiceDualPipeline(device=device, evaluator=evaluator)
    processor = BatchDatasetProcessor(pipeline=pipeline, evaluator=evaluator, device=device)
else:
    print(f'♻️ Tái sử dụng Pipeline & Evaluator đang có sẵn trên {device}.')

spoof_f5_dir = data_root / 'spoof_f5tts'
spoof_vn_dir = data_root / 'spoof_vieneu'
spoof_f5_dir.mkdir(parents=True, exist_ok=True)
spoof_vn_dir.mkdir(parents=True, exist_ok=True)

# Bản đồ transcript mẫu tham chiếu (kết hợp cả all_transcripts và bonafide_df)
ref_transcript_map = {**all_transcripts}
for _, row in bonafide_df.iterrows():
    t_val = row.get('transcript')
    if pd.notna(t_val) and str(t_val).strip():
        ref_transcript_map[str(row['utt_id'])] = str(t_val).strip()
print(f'📖 Bản đồ transcript tham chiếu có: {len(ref_transcript_map)} câu.')
all_spoof_records = []

for spk in selected_speakers:
    ckpt_file = checkpoints_dir / f'{spk}_done.json'
    spk_raw_dir = raw_out_dir / spk
    wav_files = sorted([f.as_posix() for f in spk_raw_dir.glob('*.wav')])
    if not wav_files:
        continue

    # Checkpoint & Resume: Nếu speaker đã hoàn tất trong phiên trước, đọc lại từ checkpoint
    if ckpt_file.exists():
        print(f'⏩ Speaker {spk} đã có checkpoint hoàn tất. Đang đọc lại kết quả...')
        saved_recs = json.loads(ckpt_file.read_text(encoding='utf-8'))
        all_spoof_records.extend(saved_recs)
        continue

    ref_stems = {f.stem for f in (ref_out_dir / spk).glob('*.wav')}
    spk_df = bonafide_df[bonafide_df['speaker_id'] == spk]
    non_ref_df = spk_df[~spk_df['utt_id'].isin(ref_stems)]
    raw_prompts = non_ref_df[non_ref_df['transcript'].str.len() > 10]['transcript'].tolist()[:MAX_PROMPTS_PER_SPK]
    prompts = [p.strip().lower() for p in raw_prompts]
    if not prompts:
        prompts = [
            'xin chào tôi là người nhà cần liên hệ gấp về việc viện phí.',
            'alo bạn ơi chuyển khoản giúp tôi mười triệu vào tài khoản nhé.',
            'chào bạn tôi gọi từ ngân hàng xác nhận giao dịch bảo hiểm.',
            'hôm nay trời đẹp và chúng ta cùng đi dạo trên bờ sông nhé.',
            'anh gọi lại ngay cho em khi nhận được tin nhắn này nhé.'
        ][:MAX_PROMPTS_PER_SPK]

    ref_files = sorted([f.as_posix() for f in (ref_out_dir / spk).glob('*.wav')])
    spk_source = 'common_voice' if spk.startswith('CV_') else 'vivos'
    print(f'\n🎙️ Đang xử lý {spk} ({spk_source}, {len(prompts)} câu thoại, lọc Hard Gate 1/3, gán tầng Gate 2/4)...')
    records = processor.process_speaker(
        speaker_id=spk,
        speaker_wav_files=wav_files,
        ref_cands=ref_files,
        prompts=prompts,
        out_synth_dir=data_root,
        mode=PipelineMode.DUAL_OUTPUT,
        apply_telephony=True,
        only_pass=True,
        transcript_map=ref_transcript_map,
        max_retries=2,
        out_f5_dir=spoof_f5_dir / spk,
        out_vn_dir=spoof_vn_dir / spk
    )

    # Lưu checkpoint cho speaker
    serialized = []
    for r in records:
        rec_dict = {
            'utt_id': Path(r.audio_path).stem,
            'audio_path': r.audio_path,
            'speaker_id': r.speaker_id,
            'source': spk_source,
            'label': 'spoof',
            'role': 'eval',
            'tts_generator': r.tts_generator or (r.engine_name if hasattr(r, 'engine_name') else 'F5-TTS'),
            'codec': r.codec,
            'split': 'train' if spk in train_spks else ('dev' if spk in dev_spks else 'test'),
            'duration_sec': 4.0,
            'transcript': '',
            'raw_wer': r.raw_wer,
            'raw_cer': r.raw_cer,
            'ecapa_sim': r.ecapa_sim,
            'wavlm_sim': getattr(r, 'wavlm_sim', None),
            'derived_tier': r.derive_attack_tier(),
            'seed': r.seed if r.seed is not None else 100,
        }
        serialized.append(rec_dict)

    ckpt_file.write_text(json.dumps(serialized, indent=2, ensure_ascii=False), encoding='utf-8')
    all_spoof_records.extend(serialized)
    print(f'✔️ Hoàn tất {spk}: sinh {len(serialized)} mẫu spoof (F5-TTS + VieNeu-TTS x Studio + G.711 + AMR-NB)')

spoof_df = pd.DataFrame(all_spoof_records)
spoof_csv = manifests_dir / 'spoof_manifest.csv'
spoof_df.to_csv(spoof_csv, index=False, encoding='utf-8')
print(f'\n✅ Đã hoàn tất sinh spoof! Lưu manifest tại: {spoof_csv} ({len(spoof_df)} mẫu)')


In [ ]:
# 5. Phân chia tập Speaker-Disjoint & tạo train.csv, dev.csv, test_indomain.csv (17 cột chuẩn)
MANIFEST_COLUMNS = [
    'utt_id', 'audio_path', 'speaker_id', 'source', 'label', 'role',
    'tts_generator', 'codec', 'split', 'duration_sec', 'transcript',
    'raw_wer', 'raw_cer', 'ecapa_sim', 'wavlm_sim', 'derived_tier', 'seed'
]

bonafide_std_df = bonafide_df[MANIFEST_COLUMNS].copy() if not bonafide_df.empty else pd.DataFrame(columns=MANIFEST_COLUMNS)
spoof_std_df = pd.DataFrame(all_spoof_records)[MANIFEST_COLUMNS].copy() if all_spoof_records else pd.DataFrame(columns=MANIFEST_COLUMNS)

dfs_to_concat = [df for df in [bonafide_std_df, spoof_std_df] if not df.empty]
full_df = pd.concat(dfs_to_concat, ignore_index=True) if dfs_to_concat else pd.DataFrame(columns=MANIFEST_COLUMNS)

full_manifest_csv = manifests_dir / 'full_manifest.csv'
full_df.to_csv(full_manifest_csv, index=False, encoding='utf-8')

train_df = full_df[full_df['split'] == 'train'].copy().reset_index(drop=True)
dev_df = full_df[full_df['split'] == 'dev'].copy().reset_index(drop=True)
test_df = full_df[full_df['split'] == 'test'].copy().reset_index(drop=True)

train_df.to_csv(manifests_dir / 'train.csv', index=False, encoding='utf-8')
dev_df.to_csv(manifests_dir / 'dev.csv', index=False, encoding='utf-8')
test_df.to_csv(manifests_dir / 'test_indomain.csv', index=False, encoding='utf-8')

print('=' * 60)
print('📊 THỐNG KÊ MANIFEST CHUẨN 17 CỘT (SPEAKER-DISJOINT):')
print(f" - Train ({train_df['speaker_id'].nunique()} spks): {len(train_df)} mẫu (Bona: {(train_df['label']=='bonafide').sum()}, Spoof: {(train_df['label']=='spoof').sum()})")
print(f" - Dev   ({dev_df['speaker_id'].nunique()} spks): {len(dev_df)} mẫu (Bona: {(dev_df['label']=='bonafide').sum()}, Spoof: {(dev_df['label']=='spoof').sum()})")
print(f" - Test  ({test_df['speaker_id'].nunique()} spks): {len(test_df)} mẫu (Bona: {(test_df['label']=='bonafide').sum()}, Spoof: {(test_df['label']=='spoof').sum()})")
print(f" - Tổng cộng: {len(full_df)} mẫu")
print('\nPhân bổ tầng tấn công:')
print(full_df['derived_tier'].value_counts())
print('\nPhân bổ mã hóa (Codec):')
print(full_df['codec'].value_counts())
print('\nPhân bổ nguồn dữ liệu (Source):')
print(full_df['source'].value_counts())
print('\nPhân bổ công cụ sinh (Generator):')
print(full_df['tts_generator'].value_counts())
print('=' * 60)


In [ ]:
# 6. Đóng gói thư mục data thành file ZIP cho Kaggle Output
import zipfile
import shutil

for m in manifests_dir.glob('*.csv'):
    shutil.copy(m, working_dir / m.name)

zip_out = working_dir / 'vitruevoice_data_processed.zip'
print(f'📦 Đang nén thư mục data vào {zip_out.name}...')
with zipfile.ZipFile(zip_out, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in data_root.rglob('*'):
        if f.is_file():
            zf.write(f, arcname=f.relative_to(working_dir))

print(f'✅ Đã tạo gói dữ liệu thành công: {zip_out} ({zip_out.stat().st_size / 1e6:.2f} MB)')
print('Tải kết quả về máy local: python manage_kaggle.py output')
